# Sports Facility Booking Portal

## by  (M.Mohsin) & (M.Hashim)

## Main Features:
- Student Login with Roll Number (new students are registered automatically)
- Book a Facility Slot (with double-booking check)
- View My Bookings
- Admin Login (password protected)
- Admin: View All Bookings
- Admin: View All Students
- Admin: Generate Booking Report
- All data saved in text files (users.txt and bookings.txt)

### Define Lists and Settings

In [1]:
FACILITIES = ["Badminton", "Basketball", "Futsal", "Table Tennis"]
SLOTS = ["3-4 PM", "4-5 PM", "5-6 PM", "6-7 PM"]
ADMIN_PASSWORD = "admin123"

Explanation: FACILITIES and SLOTS are lists that hold the four sports facilities and the four one-hour time slots. ADMIN_PASSWORD stores the admin password. A booking is saved using the position (0, 1, 2 or 3) of the facility and slot in these lists, so every saved line stays short and the same length. Keeping these at the top means we only change them in one place.

### Read Lines Function

In [2]:
def read_lines(filename):
    """Read a text file and return its lines as a list (empty list if it cannot be read)."""
    lines = []
    try:
        file = open(filename, "r")
        for line in file:
            lines.append(line)
        file.close()
    except FileNotFoundError:
        print("(" + filename + " not created yet)")
    except PermissionError:
        print("No permission to read", filename)
    return lines

Explanation: The function opens a text file, goes through it line by line with a for loop, adds every line to a list and returns that list. The assignment says the portal must read old records when it starts, and this function does that for both users.txt and bookings.txt. try-except handles FileNotFoundError (file not created yet) and PermissionError (no access), so the program never crashes. It only prints a message and returns an empty list.

### Append Line Function

In [3]:
def append_line(filename, text):
    """Add one new line at the end of a file."""
    file = None
    try:
        file = open(filename, "a")
        file.write(text + "\n")
    except PermissionError:
        print("No permission to write", filename)
    finally:
        if file:
            file.close()

Explanation: The function opens a file in "a" (append) mode and writes one new line at the end, so old records are never deleted. We use it to save new students and new bookings. except PermissionError handles a file we are not allowed to write, and finally always closes the file, whether an error happened or not.

### User Login Function

In [4]:
def user_login(roll):
    """Return the student's name if the roll number is registered, else an empty string."""
    for line in read_lines("users.txt"):
        if line[0:8] == roll:
            return line[8:len(line) - 1]
    return ""

Explanation: The function takes a roll number as a parameter and loops through the lines of users.txt. Each line starts with the 8 character roll number (line[0:8]) and the rest of the line is the name. If the roll number matches, the function returns the name. If not, it returns an empty string. main() uses this to know if the student is already registered.

### Add Booking Function

In [5]:
def add_record(roll, facility, slot, date):
    """Save a booking (facility and slot are list positions). Returns False if already booked."""
    record = roll + str(facility) + str(slot) + date
    for line in read_lines("bookings.txt"):
        if line[8:20] == record[8:20]:
            return False
    append_line("bookings.txt", record)
    return True

Explanation: The function builds one record: roll number + facility number + slot number + date. Before saving, a for loop checks all old bookings. If one has the same facility, slot and date (line[8:20] == record[8:20]), the function returns False because the slot is already taken. This is our double-booking check. Otherwise it saves the record with append_line() and returns True.

### Search Booking Function

In [6]:
def search_record(roll):
    """Return all bookings that belong to a roll number."""
    found = []
    for line in read_lines("bookings.txt"):
        if line[0:8] == roll:
            found.append(line)
    return found

Explanation: The function takes a roll number, loops through all saved bookings and adds the ones with that roll number to a list called found. It returns that list. We use it for the My bookings option, so a student only sees their own bookings.

### Generate Report Function

In [7]:
def generate_report():
    """Return the number of bookings for each facility."""
    lines = read_lines("bookings.txt")
    report = "Total bookings: " + str(len(lines)) + "\n"
    for i in range(len(FACILITIES)):
        count = 0
        for line in lines:
            if line[8] == str(i):
                count = count + 1
        report = report + FACILITIES[i] + ": " + str(count) + "\n"
    return report

Explanation: The function reads all bookings and counts the total. Then a for loop goes through each facility and an inner for loop counts how many bookings belong to it (line[8] is the facility number). The result is built as text and returned. The admin uses it to see which facilities are booked the most.

### Menu Pick Function

In [8]:
def pick(title, options):
    """Show a numbered menu and return the chosen position. Asks again until the input is valid."""
    print("\n===== " + title + " =====")
    for i in range(len(options)):
        print(str(i + 1) + ". " + options[i])
    while True:
        try:
            n = int(input("Choose: "))
            if 1 <= n <= len(options):
                return n - 1
            print("Pick a number from the list.")
        except ValueError:
            print("Please enter a number.")

Explanation: The function prints a title and the numbered options with a for loop. Then a while True loop asks for a number. int() is inside try-except, so if the user types letters, ValueError is caught and the question is asked again. A number outside the list is also rejected. It returns the position of the chosen option. We use it for every menu, so wrong input never crashes the portal.

### Get Date Function

In [9]:
def get_date():
    """Keep asking until the user types a valid date like 20-11-2026."""
    while True:
        date = input("Date (DD-MM-YYYY): ")
        try:
            if len(date) == 10 and 0 < int(date[0:2]) <= 31 and 0 < int(date[3:5]) <= 12:
                return date
        except ValueError:
            pass
        print("Invalid date, use the format 20-11-2026.")

Explanation: The function keeps asking for a date in a while True loop. It accepts the date only if it has 10 characters, the day is between 1 and 31 and the month is between 1 and 12. int() is inside try-except, so letters in the date are caught with ValueError. The loop only stops when the date is valid and then the date is returned.

### Show Bookings Function

In [10]:
def show_bookings(bookings):
    """Print a list of saved booking lines in a readable way."""
    if len(bookings) == 0:
        print("No bookings found.")
    for line in bookings:
        print("-", line[0:8], "|", FACILITIES[int(line[8])], "|", line[10:20], "|", SLOTS[int(line[9])])

Explanation: The function prints every booking in a list. A booking is stored as numbers, so we use the stored facility and slot numbers to take the real names from the FACILITIES and SLOTS lists. If the list is empty it prints No bookings found.

### Admin Panel Function

In [11]:
def admin_panel():
    """Admin menu: view bookings, view students and see the report."""
    while True:
        choice = pick("ADMIN PANEL", ["View all bookings", "View students", "Report", "Back"])
        if choice == 0:
            show_bookings(read_lines("bookings.txt"))
        elif choice == 1:
            for line in read_lines("users.txt"):
                print("-", line[0:8], line[8:len(line) - 1])
        elif choice == 2:
            print(generate_report())
        else:
            break

Explanation: The function shows an admin menu using pick() inside a while True loop. Option 1 shows all bookings, option 2 prints all students with a for loop over users.txt, option 3 prints the report and option 4 goes back. The admin needs this to see everything saved in the portal.

### Main Menu Function

In [12]:
def main():
    """Main loop: keeps the portal running until the user types 0."""
    print("===== SPORTS FACILITY BOOKING PORTAL =====")
    while True:
        roll = input("\nRoll number, e.g. 25L-1010 (admin for admin, 0 to exit): ").upper()
        if roll == "0":
            break
        if roll == "ADMIN":
            if input("Admin password: ") == ADMIN_PASSWORD:
                admin_panel()
            else:
                print("Wrong password.")
            continue
        if len(roll) != 8:
            print("Roll number must be 8 characters.")
            continue
        name = user_login(roll)
        if name == "":
            name = input("New student! Enter your name: ")
            append_line("users.txt", roll + name)
        print("Welcome,", name)
        while True:
            choice = pick("MENU", ["Book a slot", "My bookings", "Logout"])
            if choice == 0:
                f = pick("Facility", FACILITIES)
                d = get_date()
                s = pick("Time slot", SLOTS)
                if add_record(roll, f, s, d):
                    print("Booking confirmed!")
                else:
                    print("Sorry, that slot is already booked.")
            elif choice == 1:
                show_bookings(search_record(roll))
            else:
                break
    print("Goodbye!")

Explanation: main() joins everything together. A while True loop asks for a roll number and runs until the user types 0. Typing admin asks for the password and opens the admin panel. A roll number must be 8 characters. If the student is new, we ask for the name and save it with append_line(). Then a second while True loop shows the student menu: Book a slot (facility, date and slot, then add_record()), My bookings (search_record()) and Logout. The assignment needs a console menu that keeps running until the user exits, and this function does that.

### Sample Execution and Testing

First we delete any old data files and call the functions directly with sample values.

In [13]:
import os
for name in ["users.txt", "bookings.txt"]:
    if os.path.exists(name):
        os.remove(name)

print("TEST 1 - reading a file that does not exist (no crash):")
print(read_lines("bookings.txt"))

print("\nTEST 2 - registering two students and logging in:")
append_line("users.txt", "25L-1010Ali Raza")
append_line("users.txt", "25L-1011Sara Khan")
print(user_login("25L-1010"))
print(repr(user_login("25L-9999")), "<- empty text means not registered")

print("\nTEST 3 - add_record() and the double-booking check:")
print(add_record("25L-1010", 0, 0, "20-11-2026"))   # True  - saved
print(add_record("25L-1011", 0, 0, "20-11-2026"))   # False - same slot already booked
print(add_record("25L-1011", 0, 1, "20-11-2026"))   # True  - different slot
print(add_record("25L-1010", 2, 2, "21-11-2026"))   # True  - different facility

print("\nTEST 4 - search_record() by roll number:")
show_bookings(search_record("25L-1010"))
show_bookings(search_record("25L-1099"))

print("\nTEST 5 - generate_report():")
print(generate_report())

TEST 1 - reading a file that does not exist (no crash):
(bookings.txt not created yet)
[]

TEST 2 - registering two students and logging in:
Ali Raza
'' <- empty text means not registered

TEST 3 - add_record() and the double-booking check:
(bookings.txt not created yet)
True
False
True
True

TEST 4 - search_record() by roll number:
- 25L-1010 | Badminton | 20-11-2026 | 3-4 PM
- 25L-1010 | Futsal | 21-11-2026 | 5-6 PM
No bookings found.

TEST 5 - generate_report():
Total bookings: 3
Badminton: 2
Basketball: 0
Futsal: 1
Table Tennis: 0



Explanation: We tested each function on its own. TEST 1 reads a missing file and shows that no crash happens. TEST 2 saves two students and checks the login. TEST 3 shows that the same facility, date and slot cannot be booked twice (False). TEST 4 searches by roll number, and a roll number with no bookings gives No bookings found. TEST 5 prints the report.

In [14]:
for name in ["users.txt", "bookings.txt"]:
    print("---", name, "---")
    file = open(name, "r")
    print(file.read())
    file.close()

--- users.txt ---
25L-1010Ali Raza
25L-1011Sara Khan

--- bookings.txt ---
25L-10100020-11-2026
25L-10110120-11-2026
25L-10102221-11-2026



Explanation: This prints the two data files to show that they were created and that the records were saved. Each booking is one line: roll number (8) + facility number (1) + slot number (1) + date (10).

### Sample Run of the Full Menu

`main()` normally waits for the user to type. To show a full run here, the next cell gives ready-made answers to `input()` and prints each answer after its prompt, as if a user typed it. We typed wrong inputs on purpose (letters instead of a number, an invalid date, a short roll number, a wrong admin password).

In [15]:
answers = [
    "25L-1012", "Hamza Ali",              # new student registers
    "abc", "1", "1", "40-13-2026", "21-11-2026", "2",    # wrong menu input, then book Badminton (bad date first)
    "1", "1", "21-11-2026", "2",          # same slot again -> refused
    "2", "3",                             # my bookings, logout
    "25L-12",                             # roll number too short
    "admin", "wrong",                     # wrong admin password
    "admin", "admin123", "1", "2", "3", "4",   # admin: bookings, students, report, back
    "0",                                  # exit
]

def fake_input(prompt):
    answer = answers.pop(0)
    print(prompt + answer)
    return answer

input = fake_input      # temporarily replace input()
try:
    main()
finally:
    del input           # give back the real input()

===== SPORTS FACILITY BOOKING PORTAL =====

Roll number, e.g. 25L-1010 (admin for admin, 0 to exit): 25L-1012
New student! Enter your name: Hamza Ali
Welcome, Hamza Ali

===== MENU =====
1. Book a slot
2. My bookings
3. Logout
Choose: abc
Please enter a number.
Choose: 1

===== Facility =====
1. Badminton
2. Basketball
3. Futsal
4. Table Tennis
Choose: 1
Date (DD-MM-YYYY): 40-13-2026
Invalid date, use the format 20-11-2026.
Date (DD-MM-YYYY): 21-11-2026

===== Time slot =====
1. 3-4 PM
2. 4-5 PM
3. 5-6 PM
4. 6-7 PM
Choose: 2
Booking confirmed!

===== MENU =====
1. Book a slot
2. My bookings
3. Logout
Choose: 1

===== Facility =====
1. Badminton
2. Basketball
3. Futsal
4. Table Tennis
Choose: 1
Date (DD-MM-YYYY): 21-11-2026

===== Time slot =====
1. 3-4 PM
2. 4-5 PM
3. 5-6 PM
4. 6-7 PM
Choose: 2
Sorry, that slot is already booked.

===== MENU =====
1. Book a slot
2. My bookings
3. Logout
Choose: 2
- 25L-1012 | Badminton | 21-11-2026 | 4-5 PM

===== MENU =====
1. Book a slot
2. My bookin

Explanation: The output shows the whole portal working: a new student registers, a wrong menu input is rejected, an invalid date is asked again, a booking is confirmed and the same slot is refused the second time. Then the admin password is checked, and the admin sees all bookings, all students and the report.

### Run the Portal Yourself

In [ ]:
main()